# Auto-Differentiable Equadratures

The `equadratures.jax` namespace is a JAX-native, **differentiable** backend for `equadratures`. Quadrature, orthogonal-polynomial approximation, uncertainty quantification and learnable polynomial kernels are all `jax.grad`-able, `jax.jit`-able and `jax.vmap`-able. It is a *parallel namespace* &mdash; the classic NumPy API is unchanged.

Install with `pip install equadratures[jax]` (add `equadratures[jax-learn]` for the Optax-based kernel training used at the end).

In [ ]:
import equadratures.jax as eqj
import jax
import jax.numpy as jnp

## 1. Differentiable Gauss quadrature

Nodes and weights are the Golub&ndash;Welsch eigendecomposition of the Jacobi matrix. Because `jnp.linalg.eigh` is differentiable, so are they. An $n$-point rule integrates polynomials up to degree $2n-1$ exactly.

In [ ]:
alpha, beta, mu0 = eqj.legendre_recurrence(8)   # uniform on [-1, 1]
nodes, weights = eqj.gauss_quadrature(alpha, beta, mu0)
float(jnp.sum(weights * nodes**4))              # integral of x^4 = 0.4

For an arbitrary distribution, discretise its measure and use the Stieltjes recurrence &mdash; also differentiable w.r.t. the nodes and weights:

In [ ]:
a, b, m0 = eqj.stieltjes_recurrence(nodes, weights, n=6)
float(b[1])

## 2. Uncertainty quantification with `Poly`

Fit an orthonormal polynomial surrogate and read off the mean, variance and Sobol' sensitivity indices &mdash; all differentiable.

In [ ]:
rec = [eqj.uniform_recurrence(4), eqj.uniform_recurrence(4)]   # two uniform inputs
idx = eqj.total_order_indices(dimensions=2, order=2)
X, W = eqj.tensor_quadrature(rec)

f = lambda Z: 1.0 + 2.0*Z[:, 0] + 3.0*Z[:, 0]*Z[:, 1]
poly = eqj.Poly(rec, idx)
poly.fit_projection(X, f(X), W)

print('mean      =', float(poly.mean()))
print('variance  =', float(poly.variance()))
print('Sobol     =', [round(float(s), 4) for s in poly.sobol_indices()])
print('total Sob =', [round(float(t), 4) for t in poly.total_sobol_indices()])

## 3. Differentiating through the pipeline

The surrogate is a differentiable function of its inputs (and of the training data, and &mdash; via `Parameter` &mdash; of the distribution parameters). Here is the gradient of the surrogate at a point, which matches the analytic gradient $\nabla f = (2 + 3x_2,\; 3x_1)$.

In [ ]:
grad_f = jax.grad(lambda x: poly.predict(x[None, :])[0])
grad_f(jnp.array([0.3, -0.4]))          # -> [0.8, 0.9]

## 4. A learnable polynomial (Mercer) kernel

The *random polynomial kernel* $k(x, x') = \sum_k \theta_k^2\, \phi_k(x)\, \phi_k(x')$ is built on the orthonormal features $\phi_k$. It is positive semi-definite by construction and its log-spectrum is a learnable, interpretable parameter. It plugs straight into Gaussian-process regression:

In [ ]:
kernel = eqj.PolynomialKernel([eqj.uniform_recurrence(11)],
                              eqj.total_order_indices(1, 10))
Xtr = jnp.linspace(-1, 1, 20).reshape(-1, 1)
Xte = jnp.linspace(-0.95, 0.95, 50).reshape(-1, 1)
g = lambda X: jnp.sin(3.0 * X[:, 0])

log_theta = kernel.default_log_theta()
log_noise = jnp.log(1e-4)
pred = eqj.gp_predict(kernel, Xtr, g(Xtr), Xte, log_theta, log_noise)
float(jnp.mean((pred - g(Xte))**2))     # test MSE

The kernel's log-spectrum and the noise are learned by minimising the negative log marginal likelihood with any JAX optimiser (e.g. [Optax](https://optax.readthedocs.io)):

```python
import optax
params = {'log_theta': kernel.default_log_theta(), 'log_noise': jnp.log(1e-2)}
loss = lambda p: eqj.gp_nlml(kernel, Xtr, g(Xtr), p['log_theta'], p['log_noise'])

opt = optax.adam(5e-2); state = opt.init(params)
step = jax.jit(jax.value_and_grad(loss))
for _ in range(300):
    _, grads = step(params)
    updates, state = opt.update(grads, state, params)
    params = optax.apply_updates(params, updates)
```

Training drives the negative log marginal likelihood down and the test error to machine precision on this smooth target.